In [ ]:
!pip install ultralytics -q

import os
import xml.etree.ElementTree as ET
import shutil
import random
from tqdm import tqdm
import yaml

 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.2 MB/s eta 0:00:00a 0:00:01

In [ ]:
output_base_path = '/kaggle/working/rdd2020_yolo'
countries = ['Czech', 'Japan', 'India']
split_ratio = 0.8
class_name_to_id = {}

# AUTO-DETECTING THE BASE PATH
base_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if all(country in dirs for country in countries):
        base_path = root
        break

if not base_path:
    raise Exception("ERROR: Could not find the dataset folders. Is the data attached to the notebook?")

print(f"--> Successfully auto-detected dataset at: {base_path}\n")

# Build the rigid folder structure YOLO needs
for split in ['train', 'val']:
    os.makedirs(os.path.join(output_base_path, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_base_path, 'labels', split), exist_ok=True)

# Process each country
for country in countries:
    images_path = os.path.join(base_path, country, 'images')
    
    # Auto-detect if XMLs are in 'annotations' or 'annotations/xmls'
    ann_base = os.path.join(base_path, country, 'annotations')
    if os.path.exists(os.path.join(ann_base, 'xmls')):
        xmls_path = os.path.join(ann_base, 'xmls')
    else:
        xmls_path = ann_base

    xml_files = [f for f in os.listdir(xmls_path) if f.lower().endswith('.xml')]
    annotated_images = []

    for xml_file in xml_files:
        ann_path = os.path.join(xmls_path, xml_file)
        try:
            tree = ET.parse(ann_path)
            root = tree.getroot()
            filename = root.find('filename').text
            if not filename:
                filename = os.path.splitext(xml_file)[0] + '.jpg'
            annotated_images.append((filename, xml_file))
        except ET.ParseError:
            continue

    # Shuffle and split data
    random.shuffle(annotated_images)
    split_idx = int(len(annotated_images) * split_ratio)
    train_images = annotated_images[:split_idx]
    val_images = annotated_images[split_idx:]

    # Math and Translation
    for split, image_list in zip(['train', 'val'], [train_images, val_images]):
        for image_file, xml_file in tqdm(image_list, desc=f"Processing {country} {split}"):
            src_img = os.path.join(images_path, image_file)
            dst_img = os.path.join(output_base_path, 'images', split, image_file)
            dst_lbl = os.path.join(output_base_path, 'labels', split, f"{os.path.splitext(image_file)[0]}.txt")
            ann_path = os.path.join(xmls_path, xml_file)

            if not os.path.exists(src_img):
                continue

            try:
                tree = ET.parse(ann_path)
                root = tree.getroot()
            except ET.ParseError:
                continue

            size = root.find('size')
            if size is None:
                continue
            width = int(size.find('width').text)
            height = int(size.find('height').text)

            # Write the YOLO txt file
            with open(dst_lbl, 'w') as label_file:
                for obj in root.findall('object'):
                    cls_name = obj.find('name').text
                    if not cls_name: continue

                    if cls_name not in class_name_to_id:
                        class_name_to_id[cls_name] = len(class_name_to_id)
                    cls_id = class_name_to_id[cls_name]

                    bndbox = obj.find('bndbox')
                    if bndbox is None: continue
                    
                    xmin, ymin = float(bndbox.find('xmin').text), float(bndbox.find('ymin').text)
                    xmax, ymax = float(bndbox.find('xmax').text), float(bndbox.find('ymax').text)

                    # YOLO Math (Normalize to 0.0 - 1.0)
                    x_center = ((xmin + xmax) / 2) / width
                    y_center = ((ymin + ymax) / 2) / height
                    box_w = (xmax - xmin) / width
                    box_h = (ymax - ymin) / height

                    label_file.write(f"{cls_id} {x_center} {y_center} {box_w} {box_h}\n")
            
            # Copy image after label is successfully created
            shutil.copy(src_img, dst_img)

Successfully auto-detected dataset at: /kaggle/input/datasets/ziedkelboussi/rdd2020-dataset/train

Processing Czech train: 100%|██████████| 2263/2263 [00:11<00:00, 203.40it/s]
Processing Czech val: 100%|██████████| 566/566 [00:02<00:00, 202.05it/s]
Processing Japan train: 100%|██████████| 8404/8404 [00:45<00:00, 185.85it/s]
Processing Japan val: 100%|██████████| 2102/2102 [00:11<00:00, 190.52it/s]
Processing India train: 100%|██████████| 6164/6164 [00:30<00:00, 201.32it/s]
Processing India val: 100%|██████████| 1542/1542 [00:07<00:00, 201.27it/s]

In [ ]:
# Generate data.yaml configuration
yaml_path = os.path.join(output_base_path, 'data.yaml')

data = {
    'train': os.path.join(output_base_path, 'images', 'train'),
    'val': os.path.join(output_base_path, 'images', 'val'),
    'nc': len(class_name_to_id),
    'names': list(class_name_to_id.keys())
}

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("\n--- Preprocessing Complete ---")
print(f"Total Damage Classes Found: {len(class_name_to_id)}")
print("ID Mappings:")
for name, class_id in class_name_to_id.items():
    print(f"ID {class_id} : {name}")

--- Preprocessing Complete ---
Total Damage Classes Found: 10
ID Mappings:
ID 0 : D00
ID 1 : D10
ID 2 : D20
ID 3 : D40
ID 4 : D44
ID 5 : D50
ID 6 : D43
ID 7 : D01
ID 8 : D11
ID 9 : D0w0

In [ ]:
# Train YOLOv8 Medium with Augmentation
import os
import wandb
from ultralytics import YOLO

# Disable wandb to prevent login prompts
wandb.disabled = True
os.environ['WANDB_MODE'] = 'disabled'

print("\n--- Starting YOLOv8 Medium Training with Augmentation ---")

# Define paths and parameters
data_yaml = os.path.join(output_base_path, 'data.yaml')
model_type = 'yolov8m.pt'  # UPGRADED TO MEDIUM
epochs = 75                
img_size = 640

model = YOLO(model_type)

model.train(
    data=data_yaml, 
    epochs=epochs, 
    imgsz=img_size, 
    device=[0, 1],
    batch=16,
    
    # --- THE AUGMENTATION ---
    hsv_v=0.6,         # Modifies Brightness: Simulates harsh sun glare and deep, dark shadows
    hsv_s=0.5,         # Modifies Saturation: Helps simulate wet, muddy, or washed-out asphalt
    mixup=0.1,         # Overlays two images together 10% of the time to force the model to look at complex shapes
    bgr=0.1            # Flips colors (RGB to BGR) 10% of the time to break reliance on specific asphalt colors
)